# Phase 4d: Post-Synthesis SAE Filtering of Step 4c Candidates

This notebook closes the gap described in `future_ideas.md` (idea #3) and in the
paper *"Less is Enough"* (Eq. 9–10 and the algorithm box in the Appendix):
**run the SAE on the Round-2 candidates produced by Phase 4c and keep only the
queries that actually activate their target feature.**

**Input:** the Phase 4c output — `step2_queries.queries.tsv` (the candidate
queries) **and** `step2_queries.prompts.jsonl` (each candidate tagged with the
`FeatureID` it was generated for). The two files are row-aligned: row *N* of the
TSV corresponds to record *N* of the JSONL, and `collect_spans.py` emits that
same row index as `TextID`.

**SAE logic:** identical to Phase 4b — we reuse `collect_spans.py`, which mounts
the pretrained SAE on the LLaMA backbone and, for every input query, records each
activated neuron together with its **max-over-positions** activation
`g_i(x) = max_t Z_i(x, t)` (paper's reduction). Output:
`textspans_group0.tsv` with columns `NeuronID, TextID, Score, Span`.

**File handling:** same convention as Phase 4a / 4c — all heavy work happens on
the **local** Colab disk (`/content/...`); Google Drive is mounted **only as a
backup / log destination** at the end (and as the source of the 4c output).

**Two filtering methods (computed side-by-side):**

| Method | Candidate pool for feature *i* | Selection |
|---|---|---|
| **A. Per-feature** (paper) | only candidates *generated for* *i* | keep top-`m` by `g_i`, drop *i* if none clears `δ` |
| **B. Global** | **all** candidates, whoever they were generated for | keep top-`m` globally by `g_i`, drop *i* if none clears `δ` |

Method B can *rescue* a feature that A drops (when a query written for another
feature activates *i* above `δ`), at the cost of off-target content. We report the
difference so the trade-off is explicit.


## 1. Mount Drive & check GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Επιβεβαίωση ότι έχουμε GPU (T4 / L4)
!nvidia-smi

## 2. Εγκατάσταση Βιβλιοθηκών

In [ ]:
import os
!pip uninstall -y pandas numpy
!pip install -q transformers==4.43.4 accelerate==0.33.0 bitsandbytes datasets pandas==2.2.2 numpy

## 3. Σύνδεση με Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(hf_token)

## 4. Λήψη Κώδικα (FAC-Synthesis) — local clone

Same as Phase 4a / 4c: the repo is cloned to the **local** Colab disk
(`/content/FAC-Synthesis`), *not* onto Drive.

In [ ]:
%%bash
if [ ! -d "/content/FAC-Synthesis" ]; then
  git clone https://github.com/michalispsy/SLP_2026_SEMESTER_EXER.git FAC-Synthesis
  echo "✅ Repo cloned successfully."
else
  cd /content/FAC-Synthesis && git pull origin main
  echo "ℹ️ FAC-Synthesis already exists, pulled latest."
fi

## 5. 🔧 Central Path & Config Cell

**Every path and tunable lives here.** Nothing below hard-codes a path — change a
value here and re-run.

- `INPUT_*` point at the **Phase 4c output** (defaults to where Phase 4c saved its
  logs on Drive). If your 4c output lives elsewhere, edit `DRIVE_4C_DIR`.
- `WORK_DIR` is the local scratch space where the SAE runs.
- `DRIVE_4D_DIR` is the Drive backup folder written at the very end.
- `DELTA` (δ) is the activation threshold; `TOP_M` is how many queries to keep per
  surviving feature (paper default: δ = 0.0, m = 1).

In [ ]:
import os

# ----- Tunables (paper defaults: δ = 0.0, m = 1) -----------------------------
MODEL_KEY          = "llama"                       # backbone key for collect_spans
SAE_REPO           = "Zhongzhi1228/sae_llama_l16_h65536"
DELTA              = 0.0    # δ — keep a query only if g_i(x) > DELTA
COLLECT_THRESHOLD  = 0.0    # threshold passed to collect_spans (capture g_i > this);
                            # keep <= DELTA so the δ-filter is applied in pandas.
TOP_M              = 1      # keep top-m queries per surviving feature

# ----- Repo (local) ----------------------------------------------------------
REPO          = "/content/FAC-Synthesis"
INTERPRET_DIR = f"{REPO}/sae_feature_analysis/interpret_features"

# ----- INPUT: Phase 4c output (Drive = source of logged 4c output) -----------
DRIVE_4C_DIR        = "/content/drive/MyDrive/fac_synthesis/step_4/4c/log_files"
INPUT_QUERIES_TSV   = f"{DRIVE_4C_DIR}/step2_queries.queries.tsv"
INPUT_PROMPTS_JSONL = f"{DRIVE_4C_DIR}/step2_queries.prompts.jsonl"

# ----- Local working dir (all heavy I/O happens here) ------------------------
WORK_DIR        = "/content/phase4d"
LOCAL_QUERIES   = f"{WORK_DIR}/step2_queries.queries.tsv"
SAE_OUT_DIR     = f"{WORK_DIR}/4d_sae_out"
TEXTSPANS_TSV   = f"{SAE_OUT_DIR}/threshold_{COLLECT_THRESHOLD}/textspans_group0.tsv"

# Local output files (one pair per method)
OUT_PF_TSV   = f"{WORK_DIR}/4d_perfeature_filtered.queries.tsv"
OUT_PF_AUDIT = f"{WORK_DIR}/4d_perfeature_audit.jsonl"
OUT_GL_TSV   = f"{WORK_DIR}/4d_global_filtered.queries.tsv"
OUT_GL_AUDIT = f"{WORK_DIR}/4d_global_audit.jsonl"
OUT_SUMMARY  = f"{WORK_DIR}/4d_summary.json"

# ----- Drive backup (logging only, written at the end) -----------------------
DRIVE_4D_DIR = "/content/drive/MyDrive/fac_synthesis/step_4/4d"

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(SAE_OUT_DIR, exist_ok=True)

print("REPO              :", REPO)
print("INPUT_QUERIES_TSV :", INPUT_QUERIES_TSV)
print("INPUT_PROMPTS     :", INPUT_PROMPTS_JSONL)
print("WORK_DIR          :", WORK_DIR)
print("TEXTSPANS_TSV     :", TEXTSPANS_TSV)
print("DRIVE_4D_DIR      :", DRIVE_4D_DIR)
print(f"DELTA={DELTA}  COLLECT_THRESHOLD={COLLECT_THRESHOLD}  TOP_M={TOP_M}")

## 6. Patch `generator.py` για 4-bit Loading

Identical patch to Phase 4b (CACHE_DIR placeholder + 4-bit nf4 quantization), but
applied to the **local** repo so the SAE backbone fits on a T4/L4.

In [ ]:
import os

path = f"{INTERPRET_DIR}/generator.py"
with open(path) as f:
    src = f.read()

# (a) Fix CACHE_DIR placeholder
src = src.replace(
    'CACHE_DIR = "xxx/.cache/huggingface"',
    'CACHE_DIR = os.environ.get("HF_CACHE_DIR", "/root/.cache/huggingface")'
)

# (b) 4-bit quantization patch (same as Phase 4b)
old_load = '''        self._model = trf.AutoModelForCausalLM.from_pretrained(
            self._name,
            cache_dir=CACHE_DIR,
            torch_dtype=self._dtype,
            device_map=maps
        )'''

new_load = '''        from transformers import BitsAndBytesConfig
        _use_4bit = os.environ.get("FAC_USE_4BIT", "1") == "1"
        if _use_4bit and self._device != "cpu":
            _bnb = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=tc.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )
            self._model = trf.AutoModelForCausalLM.from_pretrained(
                self._name, cache_dir=CACHE_DIR,
                quantization_config=_bnb, device_map=maps)
        else:
            self._model = trf.AutoModelForCausalLM.from_pretrained(
                self._name, cache_dir=CACHE_DIR,
                torch_dtype=self._dtype, device_map=maps)'''

if new_load.split("\n")[1].strip() in src:
    print("ℹ️ generator.py already patched.")
else:
    src = src.replace(old_load, new_load)
    with open(path, "w") as f:
        f.write(src)
    print("✅ Patched generator.py (CACHE_DIR + 4-bit quantization)")

## 7. Κατέβασμα SAE Checkpoint (ίδιο με Phase 4b)

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download
import os, shutil

files = list_repo_files(SAE_REPO)
pth_files = [f for f in files if f.endswith(".pth")]
assert pth_files, "Δεν βρέθηκε .pth αρχείο στο repo!"
SAE_FILENAME = pth_files[0]
print("Θα κατεβάσω:", SAE_FILENAME)

SAE_PATH = hf_hub_download(repo_id=SAE_REPO, filename=SAE_FILENAME)
print("✅ SAE downloaded:", SAE_PATH)

# Symlink/copy to a name collect_spans can parse ({cls}_l{layer}_*.pth)
basename = os.path.basename(SAE_PATH)
if not basename.startswith(("topk_l", "sae_l", "ae_l", "topk5_l", "topk6_l", "topk7_l")):
    target_path = "/content/topk_l16_h65536.pth"
    if not os.path.exists(target_path):
        shutil.copy(SAE_PATH, target_path)
    SAE_PATH = target_path
    print("⚠️ Μετονομάστηκε σε:", SAE_PATH)
else:
    print("✅ Το όνομα ταιριάζει:", basename)

print("Τελικό SAE_PATH:", SAE_PATH)

## 8. Stage the Phase 4c output locally

Copy the 4c output (queries + prompts) from Drive to the local working dir, so the
SAE reads from local disk. We verify the two files are **row-aligned** (this is the
contract that lets us map `TextID → FeatureID`).

In [ ]:
import os, shutil, json

assert os.path.exists(INPUT_QUERIES_TSV),  f"❌ Δεν βρέθηκε: {INPUT_QUERIES_TSV}"
assert os.path.exists(INPUT_PROMPTS_JSONL), f"❌ Δεν βρέθηκε: {INPUT_PROMPTS_JSONL}"

shutil.copy(INPUT_QUERIES_TSV, LOCAL_QUERIES)

tsv_lines = [l for l in open(LOCAL_QUERIES, encoding="utf-8").read().split("\n") if l.strip()]
prompts   = [json.loads(l) for l in open(INPUT_PROMPTS_JSONL, encoding="utf-8") if l.strip()]

print(f"queries.tsv rows     : {len(tsv_lines)}")
print(f"prompts.jsonl records: {len(prompts)}")
assert len(tsv_lines) == len(prompts), "❌ Row mismatch — alignment broken!"

# sanity: first-40-char alignment
ok = sum(tsv_lines[n].split("\t")[0].strip()[:40] == str(prompts[n]["query"]).strip()[:40]
         for n in range(len(prompts)))
print(f"alignment check      : {ok}/{len(prompts)} rows match")
assert ok == len(prompts), "❌ Query text does not align row-for-row!"

n_feat = len(set(p["FeatureID"] for p in prompts))
print(f"distinct target features: {n_feat}")
print("✅ Input staged & aligned.")

## 9. Run the SAE (`collect_spans.py`) — same logic as Phase 4b

For every query, `collect_spans.py` computes each neuron's max-over-positions
activation and writes `textspans_group0.tsv` (`NeuronID, TextID, Score, Span`),
keeping up to the top-1000 queries per neuron — more than enough for our 636
candidates. We pass `COLLECT_THRESHOLD` (≤ δ) so all candidate activations above it
are captured; the actual δ-filter is applied afterwards in pandas.

In [ ]:
import os
os.environ["SAE_PATH_4D"] = SAE_PATH
os.makedirs(SAE_OUT_DIR, exist_ok=True)

# collect_spans.py <gpu> <model_key> <subgroup> <ttlgroup> --data-path .. --threshold .. --sae-path .. --out-dir ..
!cd "{INTERPRET_DIR}" && python collect_spans.py 0 {MODEL_KEY} 0 1 \
    --data-path "{LOCAL_QUERIES}" \
    --threshold {COLLECT_THRESHOLD} \
    --sae-path "$SAE_PATH_4D" \
    --out-dir "{SAE_OUT_DIR}"

print("\nExpecting output at:", TEXTSPANS_TSV)
assert os.path.exists(TEXTSPANS_TSV), "❌ collect_spans did not produce the expected textspans file."
print("✅ SAE scoring done.")

## 10. Load SAE scores + build the `TextID → FeatureID` mapping

- `spans`: the SAE output (`NeuronID, TextID, Score, Span`). `Score` is `g_i(x)`.
- `row2feature`: which feature each candidate was *generated for* (from prompts.jsonl).
- `Fmiss`: the set of missing features we tried to cover = the targets of Round 2.

In [ ]:
import pandas as pd, json

spans = pd.read_csv(TEXTSPANS_TSV, sep="\t")
spans["NeuronID"] = spans["NeuronID"].astype(int)
spans["TextID"]   = spans["TextID"].astype(int)
spans["Score"]    = spans["Score"].astype(float)
print(f"SAE rows: {len(spans)} | unique neurons fired: {spans.NeuronID.nunique()}")

prompts   = [json.loads(l) for l in open(INPUT_PROMPTS_JSONL, encoding="utf-8") if l.strip()]
row2feature = {i: int(r["FeatureID"]) for i, r in enumerate(prompts)}
row2query   = {i: str(r["query"])     for i, r in enumerate(prompts)}

feat2rows = {}
for row, f in row2feature.items():
    feat2rows.setdefault(f, []).append(row)

Fmiss = sorted(set(row2feature.values()))
print(f"Missing features targeted in Round 2 (Fmiss): {len(Fmiss)}")
print(f"Candidates per feature: {len(prompts)/len(Fmiss):.1f} avg")

## 11. Method A — Per-feature filter (the paper's filter)

For each missing feature *i*, look **only** at the candidates that were *generated
for i* (`feat2rows[i]`), take their activation of neuron *i*, keep the top-`m` whose
`g_i(x) > δ`, and **drop i entirely** if none clears the bar.

This is a *verification* of the targeted synthesis: "did the query we built for *i*
actually fire *i*?" The kept query's `design_feature` always equals `feature`.

In [ ]:
def filter_per_feature(spans, feat2rows, row2query, Fmiss, delta, top_m):
    kept, dropped = [], []
    for f in Fmiss:
        rows = feat2rows.get(f, [])
        sub = spans[(spans.NeuronID == f) & (spans.TextID.isin(rows))]
        sub = sub[sub.Score > delta].sort_values("Score", ascending=False)
        if len(sub) == 0:
            dropped.append(f)
            continue
        for _, r in sub.head(top_m).iterrows():
            row = int(r.TextID)
            kept.append({
                "feature": f,
                "covered_by_row": row,
                "design_feature": row2feature[row],   # == f by construction
                "g_i": float(r.Score),
                "query": row2query[row],
                "label": 1,
            })
    return kept, dropped

kept_pf, dropped_pf = filter_per_feature(spans, feat2rows, row2query, Fmiss, DELTA, TOP_M)
print(f"[PER-FEATURE]  features kept: {len(set(k['feature'] for k in kept_pf))}/{len(Fmiss)}"
      f"  | dropped: {len(dropped_pf)}  | queries kept: {len(kept_pf)}")

## 12. Method B — Global filter (top global queries per feature)

For each missing feature *i*, look at **every** candidate that activated neuron *i*
— regardless of which feature it was generated for — and keep the top-`m` whose
`g_i(x) > δ`. A kept query may have been *designed for a different feature*
(`cross_assigned = True`); those are the cases that distinguish B from A.

`collect_spans` already stores the top queries per neuron globally, so this is a
direct read of `spans`.

In [ ]:
def filter_global(spans, row2feature, row2query, Fmiss, delta, top_m):
    kept, dropped = [], []
    for f in Fmiss:
        sub = spans[spans.NeuronID == f]
        sub = sub[sub.Score > delta].sort_values("Score", ascending=False)
        if len(sub) == 0:
            dropped.append(f)
            continue
        for _, r in sub.head(top_m).iterrows():
            row = int(r.TextID)
            design = row2feature.get(row)
            kept.append({
                "feature": f,
                "covered_by_row": row,
                "design_feature": design,
                "g_i": float(r.Score),
                "query": row2query.get(row),
                "label": 1,
                "cross_assigned": (design != f),
            })
    return kept, dropped

kept_gl, dropped_gl = filter_global(spans, row2feature, row2query, Fmiss, DELTA, TOP_M)
n_cross = sum(k["cross_assigned"] for k in kept_gl)
print(f"[GLOBAL]       features kept: {len(set(k['feature'] for k in kept_gl))}/{len(Fmiss)}"
      f"  | dropped: {len(dropped_gl)}  | queries kept: {len(kept_gl)}")
print(f"[GLOBAL]       cross-assigned picks (query built for another feature): {n_cross}")

## 13. Compare the two methods

The interesting quantities:
- **rescued** = features the per-feature filter *dropped* but the global filter
  *kept* (only possible because some other feature's query activates them > δ).
- **cross-assigned** = global picks whose content was written for a different
  feature (the off-target-content risk).
- **unique training queries** after de-duplicating by query text (the paper's
  `S_gen` is a union, so identical queries collapse).

In [ ]:
surv_pf = set(k["feature"] for k in kept_pf)
surv_gl = set(k["feature"] for k in kept_gl)

rescued      = sorted(surv_gl - surv_pf)
only_pf      = sorted(surv_pf - surv_gl)   # expected empty (A ⊆ B on features)
uniq_pf      = len(set(k["query"] for k in kept_pf))
uniq_gl      = len(set(k["query"] for k in kept_gl))

summary = {
    "config": {"DELTA": DELTA, "COLLECT_THRESHOLD": COLLECT_THRESHOLD,
               "TOP_M": TOP_M, "MODEL_KEY": MODEL_KEY},
    "n_missing_features": len(Fmiss),
    "per_feature": {"features_kept": len(surv_pf), "features_dropped": len(dropped_pf),
                    "queries_kept": len(kept_pf), "unique_queries": uniq_pf},
    "global":      {"features_kept": len(surv_gl), "features_dropped": len(dropped_gl),
                    "queries_kept": len(kept_gl), "unique_queries": uniq_gl,
                    "cross_assigned": int(n_cross)},
    "features_rescued_by_global": len(rescued),
    "features_only_in_per_feature": len(only_pf),
}

import json
print(json.dumps(summary, indent=2))
print("\nFeatures rescued by global (first 20):", rescued[:20])

## 13b. Score distribution & δ-sweep

Beyond the pass/drop counts above, this shows **how strongly** features actually
activate — the distribution of the best `g_i(x)` per feature — and **how many
features survive at each δ**, for both methods. Use this to pick a δ without
re-running the SAE.
- `pf_best[i]` = best `g_i` among the candidates *generated for i* (drives Method A).
- `gl_best[i]` = best `g_i` among *all* candidates (drives Method B).
- `NaN` = feature *i* was never activated above `COLLECT_THRESHOLD` by any relevant
  candidate (it will be dropped at every δ ≥ COLLECT_THRESHOLD).

In [ ]:
import numpy as np, pandas as pd

# Best own-candidate activation per feature (Method A) and best any-candidate (Method B)
pf_best, gl_best = {}, {}
for f in Fmiss:
    own = spans[(spans.NeuronID == f) & (spans.TextID.isin(feat2rows.get(f, [])))]
    allc = spans[spans.NeuronID == f]
    pf_best[f] = own.Score.max()  if len(own)  else np.nan
    gl_best[f] = allc.Score.max() if len(allc) else np.nan

pf_s, gl_s = pd.Series(pf_best, dtype=float), pd.Series(gl_best, dtype=float)

print("=== Distribution of best g_i per feature ===")
print("\nMethod A (per-feature) best g_i:")
print(pf_s.describe(percentiles=[.1, .25, .5, .75, .9]).round(4).to_string())
print(f"  features that never fired (NaN): {int(pf_s.isna().sum())}/{len(Fmiss)}")
print("\nMethod B (global) best g_i:")
print(gl_s.describe(percentiles=[.1, .25, .5, .75, .9]).round(4).to_string())
print(f"  features that never fired (NaN): {int(gl_s.isna().sum())}/{len(Fmiss)}")

# Histogram of the per-feature best score (Method A)
bins = [-np.inf, 0, 0.5, 1.0, 1.5, 2.0, 4.0, np.inf]
hist, _ = np.histogram(pf_s.dropna().values, bins=bins)
print("\n=== Histogram of Method-A best g_i (binned) ===")
for lo, hi, c in zip(bins[:-1], bins[1:], hist):
    print(f"  ({lo:>5} , {hi:>5}] : {c:>4}  {'#' * int(40 * c / max(hist.max(),1))}")

# δ-sweep: how many features survive at each threshold
print("\n=== δ-sweep — surviving features (no SAE re-run) ===")
print(f"{'delta':>6} | {'per-feature':>12} | {'global':>8} | {'rescued':>8}")
for d in [0.0, 0.5, 1.0, 1.5, 2.0, 4.0]:
    n_pf = int((pf_s > d).sum())
    n_gl = int((gl_s > d).sum())
    print(f"{d:>6} | {n_pf:>12} | {n_gl:>8} | {n_gl - n_pf:>8}")

## 14. Write outputs (local)

For each method we write:
- a **training TSV** (`query \t label`) de-duplicated by query — this is the file
  Phase 5 consumes (drop-in replacement for the unfiltered 4c TSV);
- an **audit JSONL** with full provenance (feature, row, design_feature, g_i,
  cross_assigned).

In [ ]:
import json

def write_outputs(kept, tsv_path, audit_path):
    seen = set()
    with open(tsv_path, "w", encoding="utf-8") as ft:
        for k in kept:
            q = k["query"].replace("\t", " ").replace("\n", " ").replace("\r", " ").strip()
            if q in seen:
                continue
            seen.add(q)
            ft.write(f"{q}\t{k['label']}\n")
    with open(audit_path, "w", encoding="utf-8") as fa:
        for k in kept:
            fa.write(json.dumps(k, ensure_ascii=False) + "\n")
    return len(seen)

n_pf = write_outputs(kept_pf, OUT_PF_TSV, OUT_PF_AUDIT)
n_gl = write_outputs(kept_gl, OUT_GL_TSV, OUT_GL_AUDIT)
with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"✅ per-feature : {n_pf} unique queries -> {OUT_PF_TSV}")
print(f"✅ global      : {n_gl} unique queries -> {OUT_GL_TSV}")
print(f"✅ summary      -> {OUT_SUMMARY}")

## 15. Backup to Google Drive (logging only)

As in Phase 4a / 4c, Drive is touched **only here**, to persist the results. All
computation above ran on local disk.

In [ ]:
import os, shutil

os.makedirs(DRIVE_4D_DIR, exist_ok=True)
for p in [OUT_PF_TSV, OUT_PF_AUDIT, OUT_GL_TSV, OUT_GL_AUDIT, OUT_SUMMARY, TEXTSPANS_TSV]:
    if os.path.exists(p):
        shutil.copy(p, os.path.join(DRIVE_4D_DIR, os.path.basename(p)))
        print("backed up:", os.path.basename(p))

print("\n✅ ΤΕΛΟΣ Phase 4d. Filtered training sets are on Drive:")
print("  ", DRIVE_4D_DIR)
!ls -lh "{DRIVE_4D_DIR}"